# Машинное обучение, ФКН ВШЭ

# Практическое задание 7. Бустинг и бэггинг

## Общая информация
Дата выдачи: 08.12.2020

Мягкий дедлайн: 19.12.2020 00:59 MSK

Жёсткий дедлайн: 21.12.2020 00:59 MSK

## Оценивание и штрафы

Каждая из задач имеет определенную «стоимость» (указана в скобках около задачи). Максимально допустимая оценка за работу — 10 баллов.

Сдавать задание после указанного срока сдачи нельзя. При выставлении неполного балла за задание в связи с наличием ошибок на усмотрение проверяющего предусмотрена возможность исправить работу на указанных в ответном письме условиях.

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов (подробнее о плагиате см. на странице курса). Если вы нашли решение какого-то из заданий (или его часть) в открытом источнике, необходимо указать ссылку на этот источник в отдельном блоке в конце вашей работы (скорее всего вы будете не единственным, кто это нашел, поэтому чтобы исключить подозрение в плагиате, необходима ссылка на источник).

Неэффективная реализация кода может негативно отразиться на оценке.

## Формат сдачи
Задания сдаются через систему anytask. Посылка должна содержать:
* Ноутбук homework-practice-07-Username.ipynb

Username — ваша фамилия на латинице

## О задании

В этом задании вам предстоит вручную запрограммировать один из самых мощных алгоритмов машинного обучения — бустинг. Работать мы будем на двух наборах данных: многомерных данных по кредитам с kaggle и синтетических двумерных. В данных с kaggle целевая переменная показывает, вернуло ли кредит физическое лицо:

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression 
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, f1_score, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier

In [ ]:
df = pd.read_csv('Data/bank_data.csv')
df.sample(5)

Разделим на train и test (random_state не меняем)

In [ ]:
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
X_train_real = df_train.select_dtypes(['int64', 'float64']).drop(columns='y').values
y_train_real = df_train.y.values
X_test_real = df_test.select_dtypes(['int64', 'float64']).drop(columns='y').values
y_test_real = df_test.y.values

Генерируем синтетические данные (seed не меняем)

In [ ]:
np.random.seed(42)

num_obs = 10 ** 5
num_thresholds = 50

X_synthetic = np.random.normal(scale=3, size=[num_obs, 2])
x1_thresholds = np.random.choice(X_synthetic[:, 0], num_thresholds, False)
x2_thresholds = np.random.choice(X_synthetic[:, 1], num_thresholds, False)

gains = np.random.uniform(-0.4086, 0.5, size=[2 * num_thresholds, 1])
x1_thresholds_cond = [X_synthetic[:, 0] >= threshold for threshold in x1_thresholds]
x2_thresholds_cond = [X_synthetic[:, 1] >= threshold for threshold in x2_thresholds]

noise = np.random.uniform(-0.5, 0.5, size=num_obs)

y_synthetic_probits = np.sum(
    gains[:num_thresholds] * x1_thresholds_cond + gains[num_thresholds:] * x2_thresholds_cond, axis=0
) + noise
y_synthetic = np.sign(y_synthetic_probits)

X_train_synthetic, y_train_synthetic = X_synthetic[:int(num_obs * 0.8)], y_synthetic[:int(num_obs * 0.8)]
X_test_synthetic, y_test_synthetic = X_synthetic[int(num_obs * 0.8):], y_synthetic[int(num_obs * 0.8):]

px.histogram(x = y_synthetic_probits, nbins=100)

Некоторый полезный код для визуализации предсказаний (пригодится позже)

#### 1. (4 балла) Реализуйте бустинг для задачи бинарной классификации.

Поскольку градиентный бустинг обучается через последовательное создание моделей, может получиться так, что оптимальная с точки зрения генерализации модель будет получена на промежуточной итерации. Обычно для контроля такого поведения в методе `fit` передается также валидационная выборка, по которой можно оценивать общее качество модели в процессе обучения (желательно делать это каждую итерацию, но если ваша имплементация слишком медленная или ваше железо не тянет, можно делать это реже). Кроме того, нет смысла обучать действительно глубокую модель на 1000 деревьев и больше, если оптимальный ансамбль получился, к примеру, на 70 итерации и в течение какого-то количества итераций не улучшился - поэтому мы также задействуем early stopping при отсутствии улучшений в течение некоторого числа итераций.

In [ ]:
import matplotlib.pyplot as plt

def plot_predicts(model, features, targets, x_lim=[-15.0, 15.0], y_lim=[-15.0, 15.0],
                  examples_density=0.01, steps=1000, num_ticks=6, title='', mode='classification'):
    '''
    Функция для визуализации предсказаний модели на двухмерной плоскости
    param model: обученная модель классификации или регрессии для двухмерных объектов
    param features: признаки выборки (a.k.a. X)
    param targets: целевая переменная выборки (a.k.a y)
    param x_lim: пределы для x
    param y_lim: пределы для y
    param examples_density: доля выборки, которая будет нарисована
    param steps: частота разбиения плоскости
    param num_ticks: число подписей на графике
    param title: заголовок графика
    param mode: режим 'classification' - вероятности положительного класса
                режим 'regression' - вещественная целевая переменная
    '''
    
    mask = np.random.choice([True, False], size=features.shape[0], 
                            p=[examples_density, 1.0 - examples_density])
    features_x = (features[mask, 0] - x_lim[0]) / (x_lim[1] - x_lim[0]) * steps
    features_y = (features[mask, 1] - y_lim[0]) / (y_lim[1] - y_lim[0]) * steps
    
    xs = np.linspace(x_lim[0], x_lim[1], steps)
    ys = np.linspace(y_lim[0], y_lim[1], steps)
    
    xs, ys = np.meshgrid(xs, ys)
    grid = np.stack([xs.flatten(), ys.flatten()], axis=1)
    if mode == 'classification':
        predicts = model.predict_proba(grid).reshape(steps, steps)
        values = (targets[mask] == 1).astype('float64')
    elif mode == 'regression':
        predicts = model.predict(grid).reshape(steps, steps)
        values = targets[mask]
    else:
        raise ValueError('Unknown mode')
    
    plt.figure(figsize=(10, 10))
    plt.imshow(predicts, origin='lower')
    plt.scatter(features_x, features_y, c=values, edgecolors='white', linewidths=1.5)
    plt.colorbar()
    
    plt.xticks(np.linspace(0, steps, num_ticks), np.linspace(x_lim[0], x_lim[1], num_ticks))
    plt.yticks(np.linspace(0, steps, num_ticks), np.linspace(y_lim[0], y_lim[1], num_ticks))
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title(title)
    plt.grid()
    plt.show()

In [ ]:
class Boosting:
    
    def __init__(
        self,
        base_model_class=DecisionTreeRegressor,
        base_model_params: dict={'max_features': 10},
        n_estimators: int=10,
        learning_rate: float=0.1,
        subsample: float=0.3,
        random_seed: int=228,
        custom_loss: list or tuple=None,
        use_best_model: bool=False,
        n_iter_early_stopping: int=None
    ):
        
        # Класс базовой модели
        self.base_model_class = base_model_class
        # Параметры для инициализации базовой модели
        self.base_model_params = base_model_params
        # Число базовых моделей
        self.n_estimators = n_estimators
        # Длина шага (которая в лекциях обозначалась через eta)
        self.learning_rate = learning_rate
        # Доля объектов, на которых обучается каждая базовая модель
        self.subsample = subsample
        # seed для бутстрапа, если хотим воспроизводимость модели
        self.random_seed = random_seed
        # Использовать ли при вызове predict и predict_proba лучшее
        # с точки зрения валидационной выборки число деревьев в композиции
        self.use_best_model = use_best_model
        # число итераций, после которых при отсутствии улучшений на валидационной выборке обучение завершается
        self.n_iter_early_stopping = n_iter_early_stopping

        # Best iteration before early stopping
        self.best_iter = None
        
        # Плейсхолдер для нулевой модели
        self.initial_model_pred = None

        # Логиты в train выборке
        self.log_odds = None
        
        # Список лоссов
        self.train_losses = []
        self.val_losses = []

        # Список для хранения весов при моделях
        self.gammas = []
        
        # Создаем список базовых моделей
        self.models = [self.base_model_class(**self.base_model_params) for _ in range(self.n_estimators)]
        
        # Если используем свою функцию потерь, ее нужно передать как список из loss-a и его производной
        if custom_loss is not None:
            self.loss_fn, self.loss_derivative = custom_loss
        else:
            self.sigmoid = lambda z: 1 / (1 + np.exp(-z))
            self.loss_fn = lambda y, z: -np.log(self.sigmoid(y * z)).mean()
            self.loss_derivative = lambda y, z: -y * self.sigmoid(-y * z)
        
        
    def _fit_new_model(self, X: np.ndarray, y: np.ndarray, y_old: np.ndarray, eval_set, n_model: int):

        model = self.models[n_model]
        s_i = -self.loss_derivative(y, y_old)
        X_train, _, y_train, _ = train_test_split(X, s_i, train_size=self.subsample, random_state=self.random_seed+n_model)
        
        model.fit(X_train, y_train)
        z_train = model.predict(X)
        z_val = model.predict(eval_set[0]) if eval_set is not None else None

        self.models[n_model] = model

        return z_train, z_val
        
        
    def _fit_initial_model(self, X, y):
        p = np.mean((y == 1))
        log_odds = 0.5 * np.log(p / (1 - p))
        self.log_odds = log_odds

        return np.full(len(y), log_odds, dtype=float)
        
        # Your code here ╰( ͡° ͜ʖ ͡° )つ──☆*:
    
    
    def _find_optimal_gamma(self, y: np.ndarray or list, old_predictions: np.ndarray,
                            new_predictions: np.ndarray, boundaries: tuple or list=(0.01, 1)):
        # Определеяем начальные лосс и оптимальную гамму
        loss, optimal_gamma = self.loss_fn(y, old_predictions), 0
        # Множество, на котором будем искать оптимальное значение гаммы
        gammas = np.linspace(*boundaries, 100)
        # Простым перебором ищем оптимальное значение
        for gamma in gammas:
            predictions = old_predictions + gamma * new_predictions
            gamma_loss = self.loss_fn(y, predictions)
            if gamma_loss < loss:
                optimal_gamma = gamma
                loss = gamma_loss
        
        return optimal_gamma
        
        
    def fit(self, X, y, eval_set=None):
        self.train_losses, self.val_losses, self.gammas = [], [], []
        
        z_1 = self._fit_initial_model(X, y)
        y = np.array(y, dtype='float64').copy()
        y_pred_train = z_1
        
        if eval_set is not None:
            y_pred_val = np.full(eval_set[0].shape[0], self.log_odds, dtype='float64')
        
        best_val_loss = np.inf
        bad_iters = 0 # iterations without improving

        for i in range(self.n_estimators):
            
            z_train, z_val = self._fit_new_model(X, y, y_pred_train, eval_set, i)
            
            self.gammas.append(self._find_optimal_gamma(y, y_pred_train, z_train))
            y_pred_train += self.learning_rate * self.gammas[i] * z_train
            y_pred_val += self.learning_rate * self.gammas[i] * z_val
            
            train_loss = self.loss_fn(y, y_pred_train)
            if eval_set is not None:
                val_loss = self.loss_fn(eval_set[1], y_pred_val)
            
            self.train_losses.append(train_loss)
            self.val_losses.append(val_loss)

            # Early stopping realization
            if self.use_best_model:
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                else:
                    bad_iters += 1
                
                if bad_iters == self.n_iter_early_stopping:
                    self.best_iter = i - self.n_iter_early_stopping
                    print(f'Early stopping at iteration {i + 1}')
                    break
            
            print(f'Model {i + 1}/{self.n_estimators}, loss_train: {train_loss:.4f}, loss_val: {val_loss:.4f}')
        
        return self
        
        
    def predict(self, X: np.ndarray):
        probs = self.predict_proba(X)
        return np.where(probs > 0.5, 1, -1)
            
    def predict_proba(self, X: np.ndarray):
        z = np.full(X.shape[0], self.log_odds, dtype='float64')
        
        if self.use_best_model and self.best_iter is not None:
            num_pred = self.best_iter + 1
        else:
            num_pred = self.n_estimators

        for i in range(num_pred):
            z += self.learning_rate * self.gammas[i] * self.models[i].predict(X)
        
        return self.sigmoid(z)
        
    @property
    def feature_importances_(self):
        """
        Для бонусного задания номер 5.
        Функция для вычисления важностей признаков.
        Вычисление должно проводиться после обучения модели
        и быть доступно атрибутом класса. 
        """
        # Your code here ╰( ͡° ͜ʖ ͡° )つ──☆*:

In [ ]:
def try_base_model(
                    X_train,
                    y_train,
                    X_test,
                    y_test,
                    base_model_class=DecisionTreeRegressor,
                    base_model_params={
                        'max_features': None,
                        'max_depth': 4,
                        'min_samples_leaf': 1
                    },
                    learning_rate=0.1,
                    subsample=0.3,
                    n_estimators=100,
                    use_best_model=True,
                    n_iter_early_stopping=15
):
    X_train_s, X_val, y_train_s, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42)

    boosting = Boosting(
            base_model_class=base_model_class,
            base_model_params=base_model_params,
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            subsample=subsample,
            use_best_model=use_best_model,
            n_iter_early_stopping=n_iter_early_stopping
    )

    boosting.fit(X_train_s, y_train_s, eval_set=(X_val, y_val))
    preds = boosting.predict(X_test)

    plt.figure(figsize=(6, 4))
    iterations = [i for i in range(len(boosting.train_losses))]
    plt.plot(iterations, boosting.train_losses, label='train loss')
    plt.plot(iterations, boosting.val_losses, label='val loss')
    plt.xlabel('Iteration')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.title(f'Accuracy: {accuracy_score(y_test, preds):.4f}')
    plt.show()

    return boosting

#### Тест для вашей имплементации. Если класс написан правильно, две следующие ячейки должна отработать без ошибок и относительно быстро (у автора задания 2 и 0.2 секунд соответственно, accuracy 0.911 и 0.879 соответственно). Если у вас получилось качество выше указанного — отлично!

In [ ]:
# # Decision Tree Regressor on synthetic data
# b_model_dt_synthetic = try_base_model(
#             X_train_synthetic,
#             y_train_synthetic,
#             X_test_synthetic,
#             y_test_synthetic,
#             base_model_class=DecisionTreeRegressor,
#             base_model_params={
#                 'max_features': None,
#                 'max_depth': 6,
#                 'min_samples_leaf': 1
#             },
#             learning_rate=0.1,
#             subsample=0.7,
#             n_estimators=300,
#             use_best_model=True,
#             n_iter_early_stopping=15
# )

In [ ]:
# # Decision Tree Regressor on real data
# b_model_dt_real = try_base_model(
#             df_train.select_dtypes(['int64', 'float64']).drop(columns='y').values,
#             df_train.y.values,
#             df_test.select_dtypes(['int64', 'float64']).drop(columns='y').values,
#             df_test.y.values,
#             base_model_class=DecisionTreeRegressor,
#             base_model_params={
#                 'max_features': None,
#                 'max_depth': 6,
#                 'min_samples_leaf': 1
#             },
#             learning_rate=0.05,
#             subsample=0.7,
#             n_estimators=500,
#             use_best_model=True,
#             n_iter_early_stopping=15
# )

In [ ]:
# # Random Forest on synthetic data
# b_model_rf_synthetic = try_base_model(
#             X_train_synthetic,
#             y_train_synthetic,
#             X_test_synthetic,
#             y_test_synthetic,
#             base_model_class=RandomForestRegressor,
#             base_model_params={
#                 'n_estimators': 50,
#                 'max_depth': 5,
#                 'max_features': 'sqrt'
#             },
#             learning_rate=0.5,
#             subsample=0.7,
#             n_estimators=100,
#             use_best_model=True,
#             n_iter_early_stopping=5
# )

In [ ]:
# # Random Forest on real data
# b_model_rf_real = try_base_model(
#             df_train.select_dtypes(['int64', 'float64']).drop(columns='y').values,
#             df_train.y.values,
#             df_test.select_dtypes(['int64', 'float64']).drop(columns='y').values,
#             df_test.y.values,
#             base_model_class=RandomForestRegressor,
#             base_model_params={
#                 'n_estimators': 150,
#                 'max_depth': 5,
#                 'max_features': 'sqrt'
#             },
#             learning_rate=0.5,
#             subsample=0.7,
#             n_estimators=100,
#             use_best_model=True,
#             n_iter_early_stopping=5
# )

In [ ]:
# # Linear Regression on real data
# b_model_lr_real = try_base_model(
#             df_train.select_dtypes(['int64', 'float64']).drop(columns='y').values,
#             df_train.y.values,
#             df_test.select_dtypes(['int64', 'float64']).drop(columns='y').values,
#             df_test.y.values,
#             base_model_class=LinearRegression,
#             base_model_params={
#                 'fit_intercept': True
#             },
#             learning_rate=0.1,
#             subsample=0.9,
#             n_estimators=1000,
#             use_best_model=True,
#             n_iter_early_stopping=15
# )

In [ ]:
# # Linear Regression on synthetic data
# b_model_lr_synthetic = try_base_model(
#             X_train_synthetic,
#             y_train_synthetic,
#             X_test_synthetic,
#             y_test_synthetic,
#             base_model_class=LinearRegression,
#             base_model_params={
#                 'fit_intercept': True
#             },
#             learning_rate=0.1,
#             subsample=0.9,
#             n_estimators=200,
#             use_best_model=True,
#             n_iter_early_stopping=5
# )

In [ ]:
# plot_predicts(b_model_dt_synthetic, X_synthetic, y_synthetic, title='Boosting with Decision Tree as base model')

In [ ]:
# plot_predicts(b_model_rf_synthetic, X_synthetic, y_synthetic, title='Boosting with Random Forest as base model')

In [ ]:
# log_reg = LogisticRegression()
# log_reg.fit(X_train_synthetic, y_train_synthetic)
# print(f'Logistic Regression accuracy: {accuracy_score(y_test_synthetic, log_reg.predict(X_test_synthetic)):.4f}')

#### 2. (2 балла) Сравните результаты вашей имплементации бустинга с указанными ниже базовыми моделями на обоих датасетах и ответьте на вопросы. Разумеется, надо измерять качество на тестовых данных. 

Варианты для базовой модели (разумеется, не надо их программировать самостоятельно, берите нужные классы из sklearn):

- Решающее дерево глубины 6
- Случайный лес (число деревьев — на ваше усмотрение, только не слишком мало)
- Линейная регрессия

Вопросы:

1) Какая из моделей имеет оптимальное качество? С чем это связано?

2) Какая из моделей сильнее переобучается? Есть ли преимущества от использования ранней остановки и обрезания бустинга до лучшей модели?

3) Работает ли бустинг над линейными регрессиями лучше, чем одна логистическая регрессия? Как объяснить этот результат?

4) Визуализируйте предсказания моделей на синтетическом датасете (для этого можете воспользоваться вспомогательной функцией plot_predicts). Чем отличаются картинки, которые получаются у разных алгоритмов? Сделайте выводы.

In [ ]:
def test_model(model_class, model_params):

    model = model_class(**model_params)
    model.fit(X_train_synthetic, y_train_synthetic)
    print(f'Accuracy_synthetic: {accuracy_score(y_test_synthetic, model.predict(X_test_synthetic)):.4f}')

    model = model_class(**model_params)
    model.fit(X_train_real, y_train_real)
    print(f'Accuracy_real: {accuracy_score(y_test_real, model.predict(X_test_real)):.4f}')


In [ ]:
# # Random Forest Classifier
# test_model(
#     model_class=RandomForestClassifier,
#     model_params={
#         'max_depth': 10,
#         'n_estimators': 400
#     }
# )

# # Accuracy_synthetic: 0.9213
# # Accuracy_real: 0.8955

In [ ]:
# from sklearn.ensemble import BaggingClassifier

# # Bagging Classifier on trees (min_samples_leaf=1)
# test_model(
#     model_class=BaggingClassifier,
#     model_params={
#         'estimator': DecisionTreeClassifier(max_depth=10),
#         'n_estimators': 1000,
#         'n_jobs': -1,
#         # 'max_features': 1.0,
#         'random_state': 42
#     }
# )

# # Accuracy_synthetic: 0.9211
# # Accuracy_real: 0.8960


In [ ]:
# # Bagging Classifier on trees (max_features=0.6)
# test_model(
#     model_class=BaggingClassifier,
#     model_params={
#         'estimator': DecisionTreeClassifier(max_depth=10),
#         'n_estimators': 1000,
#         'n_jobs': -1,
#         'max_features': 0.7,
#         'random_state': 42
#     }
# )

# # Accuracy_synthetic: 0.9211
# # Accuracy_real: 0.8960


In [ ]:
# from sklearn.ensemble import GradientBoostingClassifier

# # Bagging Classifier on GBTD
# test_model(
#     model_class=BaggingClassifier,
#     model_params={
#         'estimator': GradientBoostingClassifier(
#             verbose=0,
#             n_estimators=500,
#             learning_rate=0.1,
#             subsample=0.7,
#             max_depth=4,
#             n_iter_no_change=15
#         ),
#         'n_estimators': 300,
#         'n_jobs': -1,
#         'max_features': 0.7,
#         'random_state': 42
#     }
# )

# # Accuracy_synthetic: 0.9197
# # Accuracy_real: 0.8901

In [ ]:
# # Bagging Classifier on log_reg
# test_model(
#     model_class=BaggingClassifier,
#     model_params={
#         'estimator': LogisticRegression(fit_intercept=True, max_iter=1000),
#         'n_estimators': 300,
#         'n_jobs': -1,
#     }
# )

# # Accuracy_synthetic: 0.9211
# # Accuracy_real: 0.8960

#### 3. (2 балла) Мы разобрались с бустингом, теперь интересно посмотреть на совсем дикие комбинации моделей. Сравните результаты следующих моделей на обоих датасетах и ответьте на вопросы. Разумеется, надо измерять качество на тестовых данных.

Используйте логистическую регрессию, случайный лес и BaggingClassifier из sklearn.

- Случайный лес
- Бэггинг на деревьях (поставьте для базовых деревьев min_samples_leaf=1)
- Бэггинг на деревьях с обучением каждого дерева на подмножестве признаков (`max_features` около 0.6 в BaggingClassifier)
- Бэггинг, у которого базовой моделью является бустинг с большим числом деревьев (> 100)
- Бэггинг на логистических регрессиях

1) Какая из моделей имеет лучшее качество? С чем это связано?

2) Какая из моделей сильнее всего переобучается? Помогает ли бустингу ранняя остановка? 

3) Исправляет ли бэггинг переобученность бустинга с большим числом деревьев?

4) Что лучше: случайный лес или бэггинг на деревьях с сэмплированием признаков?

5) Если использовать деревья в качестве базового алгоритма, что лучше — бэггинг или бустинг? С чем это связано?

#### 4. (2 балла) Сравните на этих данных любую из трёх популярных имплементаций градиентного бустинга (xgboost, lightgbm, catboost) с вашей реализацией. Подберите основные гиперпараметры (число деревьев, длина шага, глубина дерева/число листьев) для обоих методов. Получилось ли у вас победить библиотечные реализации на тестовых данных?

__Бонус (1 балл)__: современное развитие методов машинного обучения позволяет автоматизировать и оптимизировать подбор гиперпараметров. Воспользуйтесь для данной задачи одним из фреймворков для такого подбора - например, [hyperopt](https://github.com/hyperopt/hyperopt) или [optuna](https://github.com/optuna/optuna). Сравните также полученные данным методом модели с простым перебором гиперпараметров. 

In [ ]:
y_train_synthetic[y_train_synthetic == -1] = 0
y_test_synthetic[y_test_synthetic == -1] = 0

In [ ]:
# import xgboost as xgb
# import optuna

# def objective(trial):
#     param_distribution = {
#         'eta': trial.suggest_float('eta', 0.01, 0.3),
#         'max_depth': trial.suggest_int('max_depth', 3, 10),
#         'subsample': trial.suggest_float('subsample', 0.5, 1.0),
#         'n_estimators': trial.suggest_int('n_estimators', 100, 500),
#     }
    
#     model = xgb.XGBClassifier(**param_distribution, n_iter_no_change=15, verbose=0)
#     model.fit(X_train_synthetic, y_train_synthetic)
    
#     preds = model.predict(X_test_synthetic)
#     accuracy = accuracy_score(y_test_synthetic, preds)

#     return accuracy

# study = optuna.create_study(
#                             direction='maximize',
#                             sampler=optuna.samplers.TPESampler(seed=42),
#                             storage='sqlite:///optuna_study.db',
#                             study_name='xgb_hyperparameter_optimization',
#                             load_if_exists=True,
# )

# study.optimize(
#                objective,
#                n_trials=200,
#                show_progress_bar=True,
# )

In [ ]:
# import xgboost as xgb
# import optuna

# def objective(trial):
#     param_distribution = {
#         'eta': trial.suggest_float('eta', 0.01, 0.3),
#         'max_depth': trial.suggest_int('max_depth', 3, 10),
#         'subsample': trial.suggest_float('subsample', 0.5, 1.0),
#         'n_estimators': trial.suggest_int('n_estimators', 100, 500),
#     }
    
#     model = xgb.XGBClassifier(**param_distribution, n_iter_no_change=15, verbose=0)
#     model.fit(X_train_synthetic, y_train_synthetic)
    
#     preds = model.predict(X_test_synthetic)
#     accuracy = accuracy_score(y_test_synthetic, preds)

#     return accuracy

# study = optuna.create_study(
#                             direction='maximize',
#                             sampler=optuna.samplers.TPESampler(seed=42),
#                             storage='sqlite:///optuna_study.db',
#                             study_name='xgb_hyperparameter_optimization',
#                             load_if_exists=True,
# )

# study.optimize(
#                objective,
#                n_trials=200,
#                show_progress_bar=True,
# )

#### 5. (Бонус, 1 балл) В этом задании мы поговорим об интерпретации моделей.

Наша бустинговая модель способна возвращать вероятности для классов. Давайте попробуем оценить, насколько эти вероятности согласованы с реальностью. Для этого мы используем уже знакомый вам метод калибровочных кривых. Постройте калибровочные кривые для бустинговой модели и для логистической регрессии на обоих датасетах. Хорошо ли откалиброваны вероятности бустинга?

In [ ]:
boosting = Boosting(
            base_model_class=DecisionTreeRegressor,
            base_model_params={
                'max_features': None,
                'max_depth': 6,
                'min_samples_leaf': 1
            },
            n_estimators=300,
            learning_rate=0.3,
            subsample=0.7,
            use_best_model=True,
            n_iter_early_stopping=15
    )

In [ ]:
# Импорт необходимых библиотек
from sklearn.calibration import calibration_curve

X_train_s, X_val, y_train_s, y_val = train_test_split(X_train_synthetic, y_train_synthetic, test_size=0.1, random_state=42)

boosting.fit(X_train_s, y_train_s, eval_set=(X_val, y_val))
# Получаем предсказанные вероятности для положительного класса
y_pred_proba = boosting.predict_proba(X_test_synthetic)

# Вычисляем калибровочную кривую
prob_true, prob_pred = calibration_curve(y_test_synthetic, y_pred_proba, n_bins=10, strategy='uniform')

In [ ]:
# Строим калибровочную кривую
plt.figure(figsize=(5, 4))

# Калибровочная кривая
plt.plot(prob_pred, prob_true, marker='o', linewidth=2, label='Boosting')

# Идеально откалиброванная модель (диагональ)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Идеальная калибровка')

# Настройка графика
plt.xlabel('Среднее предсказанное значение вероятности', fontsize=12)
plt.ylabel('Доля положительных примеров', fontsize=12)
plt.title('Калибровочная кривая для модели Boosting', fontsize=14)
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.xlim([0, 1])
plt.ylim([0, 1])

# Добавляем гистограмму распределения предсказанных вероятностей
plt.figure(figsize=(5, 4))

# Гистограмма
plt.hist(y_pred_proba, bins=20, edgecolor='black', alpha=0.7, density=True)
plt.xlabel('Предсказанная вероятность', fontsize=12)
plt.ylabel('Плотность', fontsize=12)
plt.title('Распределение предсказанных вероятностей (Boosting)', fontsize=14)
plt.grid(True, alpha=0.3)

# Показываем оба графика
plt.show()

Бустинг также позволяет рассчитать важность признаков в данных. Для этого существует много подходов, но мы обратимся к самому простому. Поскольку наша базовая модель - это дерево из `sklearn`, мы можем вычислить важность признака отдельно для каждого дерева и усреднить, после этого нормировать значения, чтобы они суммировались в единицу (обратите внимание, что они должны быть неотрицательными - иначе вы что-то сделали не так). Проделайте это, затем нарисуйте столбчатые диаграммы важности признаков для обоих датасетов. На соседних графиках нарисуйте важность признаков для логистической регрессии, для этого используйте модули весов. Сравните графики. Что можно сказать?

In [ ]:
# Your code here ╰( ͡° ͜ʖ ͡° )つ──☆*:

Кстати, чаще всего излишние признаки могут вредить качеству бустинга. Попробуйте отфильтровать на основании диаграммы хвост наименее важных признаков и снова обучить модель (на тех же параметрах!). Стало ли лучше?

In [ ]:
# Your code here ╰( ͡° ͜ʖ ͡° )つ──☆*:

#### 6. (Бонус, 0.01 балла) Готовы ли вы в следующем году пойти ассистентом на этот курс и токсить на набор 19 года во флуде? 

Your answer: